# 1st Hidden Layer — Evaluate Little-Perturbation Checkpoints

In the 1st-layer perturbation sweeps, models trained with a *little*
perturbation tend to score better at their matched evaluation level. This
notebook isolates that effect: for each perturbation type it loads a **single
checkpoint trained at a small perturbation level** and evaluates that one model
across the **entire** perturbation sweep (eval-on-checkpoint), rather than
training a fresh model per level.

Each perturbation is evaluated twice — once for the **no-delay** model and once
for the **delay** model — using the same checkpoint level:

- **Jitter** — per-spike Gaussian jitter; default checkpoint `sigma = 5`.
- **Shift** — per-neuron Gaussian shift; default checkpoint `sigma = 5`.
- **Deletion** — per-spike deletion; default checkpoint `p_d = 0.2`.

Evaluations cover the **whole / part / norm** SHD variants. Per-section results
are written to `log_eval_on_perturbatedModel/` with a `1stLayer` tag so they
stay distinct from the 2nd-layer eval-on-checkpoint outputs.

In [ ]:
# Setup: reuse the 1st-layer training modules (model classes, data pipeline
# and evaluation routines) so the evaluation matches the original sweeps.
import sys
import json
from pathlib import Path

import numpy as np
import torch

BASE_DIR = Path.cwd()
assert (BASE_DIR / "jitter").is_dir(), (
    "Run this notebook from my_project/code/perturbation/ so the "
    "jitter / shift / deletion packages are importable."
)

for sub in ("jitter", "shift", "deletion"):
    sub_path = str((BASE_DIR / sub).resolve())
    if sub_path not in sys.path:
        sys.path.append(sub_path)

import jitter_train as jitter_mod
import shift_train as shift_mod
import deletion_train as deletion_mod

device = jitter_mod.device
print(f"Device: {device}")

# Dataset variants to evaluate (matches result_visualization_1stLayer.ipynb).
EVAL_DATASETS = ("whole", "part", "norm")

# Destination for the eval-on-checkpoint sweep results.
EVAL_LOG_DIR = Path("log_eval_on_perturbatedModel")
EVAL_LOG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Eval results dir: {EVAL_LOG_DIR.resolve()}")

## Shared evaluation helpers

`DATASET_CONFIGS`, `SIM_PARAMS` and the split logic are identical across the
three training modules, so the helpers below read them from `jitter_mod`.
`test_with_repeats` takes the perturbation level as its third positional
argument (`sigma` for jitter/shift, `p_d` for deletion), so one evaluation
routine serves all three perturbation types and both delay modes.

The only difference from the 2nd-layer notebook is the checkpoint filename:
1st-layer checkpoints carry no `_2ndLayer` infix
(`{perturbation}_{dataset}_{delay}_{token}.pt`), while the output JSON is
tagged with `1stLayer` so both layers can share `EVAL_LOG_DIR`.

In [ ]:
_TEST_LOADER_CACHE: dict[str, object] = {}


def get_test_loader(dataset_key: str):
    """Return the test DataLoader for ``dataset_key``.

    Uses the same split ranges and seed as training, so the evaluation set
    matches the one behind the original sweep results. Cached because the
    test data is identical across perturbation types and delay modes.
    """
    if dataset_key not in _TEST_LOADER_CACHE:
        cfg = jitter_mod.DATASET_CONFIGS[dataset_key]
        X, Y = jitter_mod.load_shd_data(
            cfg["mat_file"], target_T=jitter_mod.SIM_PARAMS["tSample"]
        )
        _, _, test_loader = jitter_mod.build_dataloaders(
            X, Y, batch_size=jitter_mod.BATCH_SIZE, seed=jitter_mod.SEED
        )
        _TEST_LOADER_CACHE[dataset_key] = test_loader
    return _TEST_LOADER_CACHE[dataset_key]


def load_checkpoint(module, net_class, dataset_key, ckpt_filename, use_delay):
    """Instantiate a network (delay or no-delay) and load a saved checkpoint."""
    cfg = module.DATASET_CONFIGS[dataset_key]
    net = net_class(
        input_dim=cfg["input_dim"],
        hidden_units=module.HIDDEN_UNITS,
        num_classes=module.NUM_CLASSES,
        use_delay=use_delay,
        max_delay=module.MAX_DELAY,
    ).to(module.device)
    ckpt_path = module.DATA_DIR / ckpt_filename
    state = torch.load(ckpt_path, map_location=module.device)
    net.load_state_dict(state)
    net.eval()
    return net


def evaluate_across_levels(module, net, dataset_key, eval_levels):
    """Evaluate one fixed model at every perturbation level."""
    test_loader = get_test_loader(dataset_key)
    results = {}
    for level in eval_levels:
        res = module.test_with_repeats(net, test_loader, level)
        results[level] = res
        print(f"      eval@{level}: {res['mean']:.4f} +/- {res['std']:.4f}")
    return results


def save_sweep_json(results, out_path, key_fn):
    """Serialise eval results to the same schema as the training sweeps."""
    serial = {
        key_fn(level): {
            "mean": float(d["mean"]),
            "std": float(d["std"]),
            "values": [float(v) for v in d["values"]],
        }
        for level, d in results.items()
    }
    with open(out_path, "w") as fp:
        json.dump(serial, fp, indent=2)
    print(f"  saved -> {out_path}")


def run_eval(module, net_class, perturbation, delay_tag, ckpt_token,
             eval_levels, key_fn):
    """Evaluate the ``perturbation``/``delay_tag`` checkpoint across all levels.

    Loads ``{perturbation}_{ds}_{delay_tag}_{ckpt_token}.pt`` for every
    dataset, evaluates it at each level in ``eval_levels`` and writes one
    ``{perturbation}_1stLayer_{ds}_{delay_tag}_evalon_{ckpt_token}.json`` per
    dataset to ``EVAL_LOG_DIR``.
    """
    use_delay = delay_tag == "delay"
    for dataset_key in EVAL_DATASETS:
        ckpt_file = (
            f"{perturbation}_{dataset_key}_{delay_tag}_{ckpt_token}.pt"
        )
        print(f"[{perturbation}/{delay_tag}] dataset={dataset_key} "
              f"| checkpoint={ckpt_file}")
        net = load_checkpoint(
            module, net_class, dataset_key, ckpt_file, use_delay
        )
        results = evaluate_across_levels(module, net, dataset_key, eval_levels)
        out_path = EVAL_LOG_DIR / (
            f"{perturbation}_1stLayer_{dataset_key}_{delay_tag}_"
            f"evalon_{ckpt_token}.json"
        )
        save_sweep_json(results, out_path, key_fn=key_fn)

## 1a. Jitter (per-spike) — no delay

Evaluate the **no-delay** checkpoint trained at `CHECKPOINT_SIGMA_JITTER` across
the full jitter sweep (`jitter_mod.SIGMA_VALUES`).

In [ ]:
# Jitter level of the checkpoint to evaluate (model trained at this sigma).
# Shared by the no-delay (1a) and delay (1b) sub-sections.
CHECKPOINT_SIGMA_JITTER = 5

run_eval(
    jitter_mod, jitter_mod.JitterSHDNetwork,
    perturbation="jitter", delay_tag="nodelay",
    ckpt_token=f"sigma{CHECKPOINT_SIGMA_JITTER}",
    eval_levels=jitter_mod.SIGMA_VALUES,
    key_fn=lambda lvl: str(int(lvl)),
)

## 1b. Jitter (per-spike) — delay

Same experiment on the **delay** model: evaluate the delay checkpoint trained at
`CHECKPOINT_SIGMA_JITTER` across the full jitter sweep.

In [ ]:
run_eval(
    jitter_mod, jitter_mod.JitterSHDNetwork,
    perturbation="jitter", delay_tag="delay",
    ckpt_token=f"sigma{CHECKPOINT_SIGMA_JITTER}",
    eval_levels=jitter_mod.SIGMA_VALUES,
    key_fn=lambda lvl: str(int(lvl)),
)

## 2a. Shift (per-neuron) — no delay

Evaluate the **no-delay** checkpoint trained at `CHECKPOINT_SIGMA_SHIFT` across
the full shift sweep (`shift_mod.SIGMA_VALUES`).

In [ ]:
# Shift level of the checkpoint to evaluate (model trained at this sigma).
# Shared by the no-delay (2a) and delay (2b) sub-sections.
CHECKPOINT_SIGMA_SHIFT = 5

run_eval(
    shift_mod, shift_mod.ShiftSHDNetwork,
    perturbation="shift", delay_tag="nodelay",
    ckpt_token=f"sigma{CHECKPOINT_SIGMA_SHIFT}",
    eval_levels=shift_mod.SIGMA_VALUES,
    key_fn=lambda lvl: str(int(lvl)),
)

## 2b. Shift (per-neuron) — delay

Same experiment on the **delay** model: evaluate the delay checkpoint trained at
`CHECKPOINT_SIGMA_SHIFT` across the full shift sweep.

In [ ]:
run_eval(
    shift_mod, shift_mod.ShiftSHDNetwork,
    perturbation="shift", delay_tag="delay",
    ckpt_token=f"sigma{CHECKPOINT_SIGMA_SHIFT}",
    eval_levels=shift_mod.SIGMA_VALUES,
    key_fn=lambda lvl: str(int(lvl)),
)

## 3a. Deletion (per-spike) — no delay

Evaluate the **no-delay** checkpoint trained at `CHECKPOINT_PD_DELETION` across
the full deletion sweep (`deletion_mod.PD_VALUES`).

In [ ]:
# Deletion probability of the checkpoint to evaluate (model trained at this p_d).
# Shared by the no-delay (3a) and delay (3b) sub-sections.
CHECKPOINT_PD_DELETION = 0.2
_pd_token = f"pd{int(round(CHECKPOINT_PD_DELETION * 10)):02d}"

run_eval(
    deletion_mod, deletion_mod.DeletionSHDNetwork,
    perturbation="deletion", delay_tag="nodelay",
    ckpt_token=_pd_token,
    eval_levels=deletion_mod.PD_VALUES,
    key_fn=lambda lvl: str(float(lvl)),
)

## 3b. Deletion (per-spike) — delay

Same experiment on the **delay** model: evaluate the delay checkpoint trained at
`CHECKPOINT_PD_DELETION` across the full deletion sweep.

In [ ]:
run_eval(
    deletion_mod, deletion_mod.DeletionSHDNetwork,
    perturbation="deletion", delay_tag="delay",
    ckpt_token=_pd_token,
    eval_levels=deletion_mod.PD_VALUES,
    key_fn=lambda lvl: str(float(lvl)),
)